<a href="https://colab.research.google.com/github/LUISTHOMAS77/projeto_aplicado/blob/main/PROJETO_APLICADO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import glob

# Defina o caminho onde estão seus CSVs no Drive.
# O asterisco (*) faz com que ele pegue todos os arquivos .csv da pasta.
caminho_arquivos = '/content/drive/MyDrive/dados/*.csv'
lista_arquivos = glob.glob(caminho_arquivos)

# Lista vazia para guardar os dataframes filtrados
dataframes_filtrados = []

# Variáveis para você ajustar conforme sua necessidade
coluna_para_filtrar = 'CD_OPERADORA'
valor_do_filtro = '352501'

for arquivo in lista_arquivos:
    print(f"Processando: {arquivo}")

    # Lê o CSV. Se der erro de codificação, adicione encoding='latin1' ou 'utf-8'
    df_temp = pd.read_csv(arquivo)

    # Aplica o filtro imediatamente para economizar RAM
    df_temp_filtrado = df_temp[df_temp[coluna_para_filtrar] == valor_do_filtro]

    # Adiciona o resultado filtrado na lista
    dataframes_filtrados.append(df_temp_filtrado)

# Junta todos os dataframes filtrados em um único
df_final = pd.concat(dataframes_filtrados, ignore_index=True)
print(f"Total de linhas após o filtro: {len(df_final)}")

Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_07.csv


ParserError: Error tokenizing data. C error: Expected 2 fields in line 208, saw 3


In [3]:
import pandas as pd
import glob

# Caminho dos arquivos CSV
caminho_arquivos = '/content/drive/MyDrive/dados/*.csv'
lista_arquivos = glob.glob(caminho_arquivos)

dataframes_filtrados = []

# Definição do filtro
coluna_para_filtrar = 'CD_OPERADORA'
valor_do_filtro = '352501'

for arquivo in lista_arquivos:
    print(f"Processando: {arquivo}")

    # Define o separador como ponto e vírgula (sep=';') e a codificação (encoding='latin1')
    df_temp = pd.read_csv(arquivo, sep=';', encoding='latin1', low_memory=False)

    # Converte a coluna para texto para evitar incompatibilidade de tipos
    df_temp[coluna_para_filtrar] = df_temp[coluna_para_filtrar].astype(str)

    # Aplica o filtro
    df_temp_filtrado = df_temp[df_temp[coluna_para_filtrar] == valor_do_filtro]

    dataframes_filtrados.append(df_temp_filtrado)

# Junta todos os dataframes filtrados em um único
df_final = pd.concat(dataframes_filtrados, ignore_index=True)
print(f"Total de linhas após o filtro: {len(df_final)}")

Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_07.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_01.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_02.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_03.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_04.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_05.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_06.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_08.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_09.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_10.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_11.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2025_12.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2026_01.csv
Processando: /content/drive/MyDrive/dados/pda-024-icb-RS-2026_02.csv
Processando: /content/drive/MyDriv

In [4]:
# Coluna que contém a frequência (os números inteiros)
coluna_frequencia = 'QT_BENEFICIARIO_ATIVO'

# O df.index.repeat repete o índice de cada linha X vezes, baseado no valor da coluna
df_expandido = df_final.loc[df_final.index.repeat(df_final[coluna_frequencia])]

# Reseta o índice para ficar organizado (0, 1, 2, 3...)
df_expandido = df_expandido.reset_index(drop=True)

print(f"Total de linhas após expansão: {len(df_expandido)}")

Total de linhas após expansão: 13116041


In [6]:
import re
import numpy as np
import pandas as pd


def converter_faixa_etaria(valor):
  # Trata valores nulos ou inválidos ("Informada Incorr")
  if pd.isna(valor) or 'incorr' in str(valor).lower():
    return np.nan

  texto = str(valor).strip().lower()

  # Casos limites específicos
  if 'menos de 1' in texto:
    return 0.5
  if '80' in texto and 'mais' in texto:
    return 82.5  # Estimativa para a faixa aberta de 80+

  # Extrai os dois números da faixa (ex: "20 a 24 anos" -> [20, 24])
  numeros = re.findall(r'\d+', texto)
  if len(numeros) == 2:
    inicio, fim = float(numeros[0]), float(numeros[1])
    return (inicio + fim) / 2.0

  return np.nan


# Aplicação no seu DataFrame:
# Substitua 'DS_FAIXA_ETARIA' pelo nome correto da sua coluna
df['idade_ponto_medio'] = df['DS_FAIXA_ETARIA'].apply(converter_faixa_etaria)

NameError: name 'df' is not defined

In [ ]:
# Cria uma coluna numérica simples (1 para M, 0 para F/outros)
df['TP_SEXO_M'] = (df['TP_SEXO'].str.upper() == 'M').astype(int)

In [ ]:
colunas_planos = [
    'DE_CONTRATACAO_PLANO',
    'DE_SEGMENTACAO_PLANO',
    'DE_ABRG_GEOGRAFICA_PLANO',
    'COBERTURA_ASSIST_PLANO',
    'TIPO_VINCULO',
]

# Gera os dummies mantendo apenas N-1 colunas para cada variável
df_encoded = pd.get_dummies(
    df, columns=colunas_planos, drop_first=True, dtype=int
)

In [ ]:
# Mantém os 10 municípios com mais registros e transforma os demais em 'Outros'
top_municipios = df['NM_MUNICIPIO'].value_counts().nlargest(10).index

df['NM_MUNICIPIO_agrupado'] = df['NM_MUNICIPIO'].apply(
    lambda x: x if x in top_municipios else 'Outros'
)

# Agora sim aplica o get_dummies na versão reduzida
df = pd.get_dummies(df, columns=['NM_MUNICIPIO_agrupado'], drop_first=True)

In [5]:
# Salva o resultado final em um novo CSV
caminho_salvar = '/content/drive/MyDrive/dados/dados_perfil_unimed_poa.csv'
df_expandido.to_csv(caminho_salvar, index=False)
print("Arquivo salvo com sucesso!")

Arquivo salvo com sucesso!
